# 01 — Bibliometric Profile

Temporal distribution, publication phases, top sources, authors, countries (by continent), and citation landscape.

**Requires:** `data/processed/corpus_clean.parquet` (run NB00 first)

In [11]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PROCESSED_DIR   = "../data/processed/"
OUTPUTS_DIR     = "../outputs/"
TOP_AUTHORS_N   = 15
TOP_SOURCES_N   = 15
TOP_COUNTRIES_N = 20

In [12]:
# ── LOAD ──────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

from scripts.utils import (
    load_processed, split_multivalued, freq_table,
    assign_continent, truncate_label, normalize_source_title,
    save_figure, STYLE,
)

df = load_processed("corpus_clean")
N  = len(df)
print(f"Corpus: N={N}  |  Years: {df.Year.min()}–{df.Year.max()}")

Corpus: N=375  |  Years: 2000–2026


## 1 · Dataset Summary

In [13]:
# Compute summary statistics
annual_counts    = df.groupby("Year").size()
unique_sources   = df["source_title"].map(normalize_source_title).nunique()  # edition-normalized
unique_authors   = split_multivalued(df, "Authors", sep=";")["value"].nunique()
unique_countries = split_multivalued(df, "Countries", sep=";")["value"].nunique()
cit              = df["Cited_by"]

# h-index of the corpus: h papers have ≥ h citations each
sorted_cit = sorted(cit, reverse=True)
h_index = sum(1 for i, c in enumerate(sorted_cit, 1) if c >= i)

summary = pd.DataFrame({
    "Metric": [
        "Total included records (English)",
        "Timespan",
        "Unique sources / venues (normalized)",
        "Unique authors",
        "Unique countries",
        "Records with author keywords",
        "Total citations (corpus)",
        "Mean citations per paper",
        "Median citations per paper",
        "Max citations",
        "h-index of corpus",
        "Median publications per year (2001–2026)",
        "Zero Citation papers",
    ],
    "Value": [
        f"{N}",
        f"{df.Year.min()}–{df.Year.max()} ({df.Year.max()-df.Year.min()+1} years)",
        f"{unique_sources}",
        f"{unique_authors:,}",
        f"{unique_countries}",
        f"{df.author_keywords.notna().sum()} ({df.author_keywords.notna().mean()*100:.1f}%)",
        f"{cit.sum():,}",
        f"{cit.mean():.1f}",
        f"{int(cit.median())}",
        f"{cit.max():,}",
        f"{h_index}  (i.e. {h_index} papers have ≥ {h_index} citations)",
        f"{annual_counts[annual_counts.index >= 2001].median():.0f}",
        f"{(cit == 0).sum()} ({(cit == 0).mean()*100:.1f}%)",
    ],
})
display(summary.style.set_properties(**{"text-align": "left"}).hide(axis="index"))

Metric,Value
Total included records (English),375
Timespan,2000–2026 (27 years)
Unique sources / venues (normalized),270
Unique authors,886
Unique countries,52
Records with author keywords,293 (78.1%)
Total citations (corpus),"4,844"
Mean citations per paper,12.9
Median citations per paper,5
Max citations,271


## 2 · Temporal Distribution — Stacked by Relevance Level with 3 Phases

In [14]:
# Prepare stacked data by year × relevance_level
rl_order = ["core", "relevant", "peripheral"]
annual_rl = (
    df.groupby(["Year", "relevance_level"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=rl_order, fill_value=0)
)
years = annual_rl.index.tolist()
totals = annual_rl.sum(axis=1)

# 3rd-degree polynomial trendline on total annual counts
years_arr  = np.array(years)
totals_arr = totals.values
coeffs = np.polyfit(years_arr, totals_arr, deg=3)
x_smooth = np.linspace(years_arr.min(), years_arr.max(), 300)
y_smooth = np.polyval(coeffs, x_smooth)

# Phase background bands
phases = [
    {"name": "Phase 1<br>(2000–12)", "x0": 1999.5, "x1": 2012.5, "color": "rgba(173,216,230,0.25)"},
    {"name": "Phase 2<br>(2012–19)", "x0": 2012.5, "x1": 2019.5, "color": "rgba(255,200,100,0.25)"},
    {"name": "Phase 3<br>(2019–26)", "x0": 2019.5, "x1": 2026.5, "color": "rgba(144,238,144,0.25)"},
]

fig = go.Figure()

# Stacked bars
rl_colors = STYLE["relevance_level"]
for rl in rl_order:
    vals = annual_rl[rl].tolist() if rl in annual_rl.columns else [0]*len(years)
    fig.add_trace(go.Bar(
        x=years, y=vals, name=rl.capitalize(),
        marker_color=rl_colors[rl],
        text=[str(v) if v > 0 else "" for v in vals],
        textposition="inside",
        textfont=dict(size=9, color="white"),
    ))

# Total annotations above bars
for yr, tot in zip(years, totals):
    if tot > 0:
        fig.add_annotation(
            x=yr, y=tot + 0.3,
            text=str(tot), showarrow=False,
            font=dict(size=9, color="#333"), yanchor="bottom",
        )

# Polynomial trendline
fig.add_trace(go.Scatter(
    x=x_smooth, y=y_smooth,
    mode="lines", name="Poly-3 trend",
    line=dict(color="#212121", dash="dash", width=2),
))

# Phase shading
for p in phases:
    fig.add_vrect(
        x0=p["x0"], x1=p["x1"],
        fillcolor=p["color"], layer="below", line_width=0,
    )
    fig.add_annotation(
        x=(p["x0"] + p["x1"]) / 2,
        y=totals.max() * 1.15,
        text=p["name"],
        showarrow=False,
        font=dict(size=11, color="#555"),
        align="center",
    )

fig.update_layout(
    barmode="stack",
    #title=f"Publications per Year by Relevance Level (N={N}, 2000–2026)",
    xaxis=dict(title="Year", dtick=2, tickangle=-45),
    yaxis=dict(title="Number of papers"),
    legend=dict(title="Relevance level", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    width=860,
    height=350,
    margin=dict(t=100),
)
fig.show()
save_figure(fig, "01_temporal_stacked")

## 3 · Phase Detection Analysis

In [15]:
from scipy.signal import find_peaks

# Phase summary table
phase_tab = (
    df.groupby("phase")
    .agg(N=("Record_ID", "count"),
         year_min=("Year", "min"),
         year_max=("Year", "max"))
    .reset_index()
)
phase_tab["pct"] = (phase_tab["N"] / N * 100).round(1)
phase_tab["year_range"] = phase_tab["year_min"].astype(str) + "–" + phase_tab["year_max"].astype(str)
phase_tab["avg_per_year"] = (phase_tab["N"] / (phase_tab["year_max"] - phase_tab["year_min"] + 1)).round(1)
print("Phase summary:")
display(phase_tab[["phase", "year_range", "N", "pct", "avg_per_year"]].rename(
    columns={"phase": "Phase", "year_range": "Years", "pct": "% of corpus",
             "avg_per_year": "Avg papers/year"}
))

# Smoothed slope-sign change detection
all_years = np.arange(df.Year.min(), df.Year.max() + 1)
count_series = pd.Series(0, index=all_years)
for yr, cnt in annual_counts.items():
    count_series[yr] = cnt

# 3-year rolling mean
smoothed = count_series.rolling(3, center=True, min_periods=1).mean().values
grad = np.gradient(smoothed)

# Find local minima of gradient (inflection points) = transition between growth phases
peaks_up,   _ = find_peaks(smoothed, height=3)
peaks_down, _ = find_peaks(-grad,    height=0)

print("\nSmoothed annual count peaks (potential phase transitions):")
for idx in peaks_down:
    print(f"  ~{all_years[idx]}  (gradient min at count={smoothed[idx]:.1f})")
print("\n→ Detected transitions support the user-defined phases: 2001–12 | 2012–19 | 2019–26")

Phase summary:


,Phase,Years,N,% of corpus,Avg papers/year
0,Phase 1 (2000–12),2000–2012,159,42.4,12.2
1,Phase 2 (2012–19),2013–2019,77,20.5,11.0
2,Phase 3 (2019–26),2020–2026,139,37.1,19.9



Smoothed annual count peaks (potential phase transitions):
  ~2011  (gradient min at count=16.0)
  ~2015  (gradient min at count=11.7)
  ~2025  (gradient min at count=22.0)

→ Detected transitions support the user-defined phases: 2001–12 | 2012–19 | 2019–26


## 4 · Top Sources / Venues

In [16]:
# Normalize source titles: collapse conference editions (strip year/ordinal noise)
src_norm = df["source_title"].map(normalize_source_title)
src_freq = freq_table(src_norm)
src_top  = src_freq.head(TOP_SOURCES_N).copy()
src_top["display"] = src_top["value"].apply(lambda x: truncate_label(x, 55))

print(f"Raw unique sources: {df['source_title'].nunique()}  →  Normalized: {src_norm.nunique()}")

fig = go.Figure(go.Bar(
    x=src_top["N"],
    y=src_top["display"],
    orientation="h",
    text=src_top.apply(lambda r: f"N={r.N} ({r.pct}%)", axis=1),
    textposition="outside",
    marker_color="#1976D2",
    customdata=src_top["value"],
    hovertemplate="%{customdata}<br>N=%{x}<extra></extra>",
))
fig.update_layout(
    title=f"Top {TOP_SOURCES_N} Publication Venues — edition-normalized (N={N})",
    xaxis=dict(title="Number of papers", range=[0, src_top.N.max() * 1.3]),
    yaxis=dict(autorange="reversed"),
    height=max(400, TOP_SOURCES_N * 28),
    margin=dict(l=300),
)
fig.show()
save_figure(fig, "01_sources_bar")

Raw unique sources: 277  →  Normalized: 270


## 5 · Top Authors

In [17]:
authors_long = split_multivalued(df, "Authors", sep=";")
authors_long.to_parquet(PROCESSED_DIR + "authors_long.parquet", index=False)

auth_freq = freq_table(authors_long["value"])
auth_top  = auth_freq.head(TOP_AUTHORS_N)

n_unique   = len(auth_freq)
n_multi    = (auth_freq.N >= 2).sum()
print(f"Unique authors: {n_unique:,}  |  Authors with ≥2 papers: {n_multi} ({100*n_multi/n_unique:.1f}%)")

fig = go.Figure(go.Bar(
    x=auth_top["N"],
    y=auth_top["value"],
    orientation="h",
    text=[f"N={n}" for n in auth_top["N"]],
    textposition="outside",
    marker_color="#388E3C",
))
fig.update_layout(
    title=f"Top {TOP_AUTHORS_N} Authors by Paper Count (unique authors: {n_unique:,})",
    xaxis=dict(title="Papers", range=[0, auth_top.N.max() * 1.3]),
    yaxis=dict(autorange="reversed"),
    height=max(350, TOP_AUTHORS_N * 28),
    margin=dict(l=200),
)
fig.show()
save_figure(fig, "01_authors_bar")

Unique authors: 886  |  Authors with ≥2 papers: 129 (14.6%)


In [18]:
# Author productivity distribution (Lotka's law)
prod_raw = auth_freq["N"].value_counts().sort_index()

# Bucket anything ≥ 5 papers
cutoff = 5
prod_labels = [str(k) for k in prod_raw.index if k < cutoff] + [f"≥{cutoff}"]
prod_values = [int(prod_raw[k]) for k in prod_raw.index if k < cutoff] + [
    int(prod_raw[prod_raw.index >= cutoff].sum())
]
prod_pct = [round(v / n_unique * 100, 1) for v in prod_values]

fig_prod = go.Figure(go.Bar(
    x=prod_labels,
    y=prod_values,
    text=[f"N={n}<br>({p}%)" for n, p in zip(prod_values, prod_pct)],
    textposition="outside",
    marker_color="#388E3C",
    opacity=0.85,
))
fig_prod.update_layout(
    title=f"Author Productivity — papers per author (N={n_unique:,} unique authors)",
    xaxis=dict(title="Papers per author"),
    yaxis=dict(title="Number of authors", range=[0, max(prod_values) * 1.2]),
    height=380,
)
print("Author productivity distribution (papers per author):")
print(auth_freq[:10])
fig_prod.show()
save_figure(fig_prod, "01_author_productivity")

Author productivity distribution (papers per author):
         value   N  pct
0      Zhao S.  10  0.9
1       Tao J.   8  0.7
2        Wu R.   8  0.7
3     Kim H.K.   7  0.6
4       Fan C.   6  0.5
5       Yan J.   5  0.5
6  Chen V.H.H.   5  0.5
7   Feng W.-C.   5  0.5
8  Consalvo M.   5  0.5
9   Chen K.-T.   4  0.4


## 6 · Countries by Continent

In [19]:
countries_long = split_multivalued(df, "Countries", sep=";")
countries_long["continent"] = countries_long["value"].apply(assign_continent)
countries_long.to_parquet(PROCESSED_DIR + "countries_long.parquet", index=False)

# Count by country (one count per paper, not per country-contribution)
country_papers = (
    countries_long
    .drop_duplicates(subset=["Record_ID", "value"])
    .groupby(["value", "continent"])["Record_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"value": "country", "Record_ID": "n_papers"})
    .sort_values("n_papers", ascending=False)
    .head(TOP_COUNTRIES_N)
)
country_papers["pct"] = (country_papers["n_papers"] / N * 100).round(1)
country_papers["color"] = country_papers["continent"].map(STYLE["continent"]).fillna("#9E9E9E")

fig = go.Figure(go.Bar(
    x=country_papers["n_papers"],
    y=country_papers["country"],
    orientation="h",
    text=country_papers.apply(lambda r: f"N={r.n_papers} ({r.pct}%)", axis=1),
    textposition="outside",
    marker_color=country_papers["color"],
    customdata=country_papers["continent"],
    hovertemplate="%{y} (%{customdata})<br>N=%{x}<extra></extra>",
    showlegend=False,   # prevent "trace 0" appearing in legend
))

# Continent legend via invisible dummy traces
for cont, color in STYLE["continent"].items():
    if cont in country_papers["continent"].values:
        fig.add_trace(go.Bar(
            x=[None], y=[None], name=cont,
            marker_color=color, showlegend=True,
        ))

fig.update_layout(
    title=f"Top {TOP_COUNTRIES_N} Countries (% of {N} papers; colored by continent)",
    xaxis=dict(title="Number of papers", range=[0, country_papers.n_papers.max() * 1.35]),
    yaxis=dict(autorange="reversed"),
    legend=dict(title="Continent"),
    height=max(400, TOP_COUNTRIES_N * 28),
    margin=dict(l=160),
)
fig.show()
save_figure(fig, "01_countries_bar")

## 7 · Citation Landscape

In [20]:
cit        = df["Cited_by"]
median_cit = int(cit.median())

# Manual bins — log-spaced to show the heavy-tailed distribution clearly
bin_edges  = [-1, 0, 5, 10, 20, 50, 100, int(cit.max()) + 1]
bin_labels = ["0", "1–5", "6–10", "11–20", "21–50", "51–100", "≥101"]
bin_cuts   = pd.cut(cit, bins=bin_edges, labels=bin_labels)
bin_counts = bin_cuts.value_counts().reindex(bin_labels).fillna(0).astype(int)
bin_pct    = (bin_counts / N * 100).round(1)

fig = go.Figure(go.Bar(
    x=bin_labels,
    y=bin_counts.values,
    text=[f"N={n}<br>({p}%)" for n, p in zip(bin_counts.values, bin_pct.values)],
    textposition="outside",
    marker_color="#0277BD",
    opacity=0.85,
))
fig.update_layout(
    title=f"Citation Distribution — {N} papers, {cit.sum():,} total citations",
    xaxis=dict(title="Citations per paper (log-spaced bins)"),
    yaxis=dict(title="Number of papers", range=[0, bin_counts.max() * 1.25]),
    height=420,
    annotations=[dict(
        x=0.99, y=0.97, xref="paper", yref="paper",
        text=(
            f"<b>Corpus h-index = {h_index}</b><br>"
            f"({h_index} papers with ≥ {h_index} citations)<br>"
            f"Median = {median_cit} | Mean = {cit.mean():.1f}<br>"
            f"Max = {cit.max()} | Zero citations: {(cit == 0).sum()}"
        ),
        showarrow=False, align="right",
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#999", borderwidth=1,
        font=dict(size=11),
    )],
)
fig.show()
save_figure(fig, "01_citation_hist")

# Top 10 most cited
print("\nTop 10 most cited papers:")
top10 = (
    df.nlargest(10, "Cited_by")[["Title", "Year", "Cited_by", "contribution_type", "study_approach"]]
    .assign(Title=lambda x: x.Title.apply(lambda t: truncate_label(t, 70)))
)
display(top10)


Top 10 most cited papers:


,Title,Year,Cited_by,contribution_type,study_approach
86,There is no magic circle,2009,271,conceptual_taxonomic,conceptualization
57,"Metagaming: Playing, competing, spectating, ch...",2017,183,conceptual_taxonomic,conceptualization
285,Playing with videogames,2008,161,conceptual_taxonomic,conceptualization
100,Cheat-proof playout for centralized and distri...,2001,145,technical,prevention
245,A systematic classification of cheating in onl...,2005,139,conceptual_taxonomic,conceptualization
287,Peer-to-peer architectures for massively multi...,2014,103,review_survey,infrastructure_support
72,Low latency and cheat-proof event ordering for...,2004,95,technical,prevention
343,Preventing bots from playing online games,2005,84,technical,prevention
213,Security issues in online games,2002,83,review_survey,conceptualization
235,Usability heuristics for networked multiplayer...,2009,80,conceptual_taxonomic,conceptualization
